# Toronto Condo Rental Market Analysis

This notebook is the cleaned portfolio version of the original web scraping project. The heavy scraping and cleaning logic has been moved into reusable Python modules under `src/`, while this notebook focuses on analysis and storytelling.

## 1. Project Objective

The objective is to collect Toronto condo rental listings and analyze how rental prices vary by area, property type, furnishing status, parking, bathrooms, and estimated unit size.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path("../data/processed/condos_clean.csv")
CHART_DIR = Path("../outputs/charts")
CHART_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Cleaned Data

Run `python src/scraper.py` and `python src/cleaning.py` from the project root before using the latest scraped data. If you already have a cleaned CSV, place it under `data/processed/condos_clean.csv`.

In [ ]:
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    display(df.head())
    print(f"Rows: {len(df):,}")
else:
    print("Cleaned data file not found. Add condos_clean.csv to data/processed/ or run the scraper and cleaning scripts.")

## 3. Data Quality Check

In [ ]:
if DATA_PATH.exists():
    display(df.info())
    display(df.isna().mean().sort_values(ascending=False).to_frame("missing_rate"))

## 4. Rental Price Distribution

In [ ]:
if DATA_PATH.exists():
    ax = df["price"].plot(kind="hist", bins=40, figsize=(10, 5))
    ax.set_title("Toronto Condo Rental Price Distribution")
    ax.set_xlabel("Monthly Rent")
    ax.set_ylabel("Number of Listings")
    plt.tight_layout()
    plt.savefig(CHART_DIR / "price_distribution.png", dpi=150)
    plt.show()

## 5. Listing Count by Area

In [ ]:
if DATA_PATH.exists() and "area" in df.columns:
    area_counts = df["area"].value_counts().head(15)
    ax = area_counts.plot(kind="bar", figsize=(12, 5))
    ax.set_title("Top Areas by Number of Rental Listings")
    ax.set_xlabel("Area")
    ax.set_ylabel("Number of Listings")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(CHART_DIR / "listing_count_by_area.png", dpi=150)
    plt.show()

## 6. Price vs. Unit Size

In [ ]:
if DATA_PATH.exists() and {"price", "size_sqft_estimate"}.issubset(df.columns):
    ax = df.plot(kind="scatter", x="size_sqft_estimate", y="price", alpha=0.35, figsize=(9, 5))
    ax.set_title("Rental Price vs. Estimated Unit Size")
    ax.set_xlabel("Estimated Unit Size (sqft)")
    ax.set_ylabel("Monthly Rent")
    plt.tight_layout()
    plt.savefig(CHART_DIR / "price_vs_size.png", dpi=150)
    plt.show()

## 7. Area-Level Summary

In [ ]:
if DATA_PATH.exists() and "area" in df.columns:
    area_summary = (
        df.groupby("area")
        .agg(
            listings=("price", "size"),
            median_price=("price", "median"),
            average_price=("price", "mean"),
            median_size_sqft=("size_sqft_estimate", "median"),
        )
        .sort_values("median_price", ascending=False)
        .reset_index()
    )
    display(area_summary)

## 8. Key Findings

Use this section to summarize your strongest insights after running the notebook with the latest cleaned dataset. Example framing:

- Downtown listings tend to have higher median rents than outer areas.
- Unit size is positively associated with rent, but area and property features also matter.
- Furnishing and parking can be used as additional segmentation variables when comparing rental listings.

## 9. Next Steps

Potential enhancements:

- Add geospatial analysis by neighbourhood.
- Build a rent prediction model using size, area, property type, bathrooms, parking, and furnishing status.
- Create a dashboard for interactive rental market exploration.